# Unit 4 Assignment: Evaluated Agentic RAG System

**System Overview:**  
A self-evaluating agentic RAG pipeline using CrewAI where:
1. **Agent 1 (RAG Retriever)** — retrieves relevant context from a FAISS vector store and generates an answer
2. **Agent 2 (Quality Evaluator)** — scores the answer using DeepEval's Faithfulness and Answer Relevancy metrics
3. **Agent 3 (Revisor)** — revises the answer when quality falls below threshold (0.7)

**Knowledge Base Topic:** The James Webb Space Telescope (JWST) — its design, mission objectives, scientific discoveries, and instrumentation. Chosen because it's a rich, factual domain with clear ground truth, making it ideal for testing RAG faithfulness.

## Setup & Installation

In [1]:
!pip install -q litellm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.14.3 requires pydantic~=2.11.9, but you have pydantic 2.12.5 which is incompatible.
crewai 1.14.3 requires python-dotenv<2,>=1.2.2, but you have python-dotenv 1.0.1 which is incompatible.
crewai-tools 1.14.3 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
deepeval 3.9.6 requires python-dotenv<2.0.0,>=1.1.1, but you have python-dotenv 1.0.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
gradio 5.50.0 requ

In [2]:
!pip3 install -q crewai crewai-tools langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers deepeval groq google-generativeai langchain-google-genai \
    langchain-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.13 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.13 requires python-dotenv==1.0.1, but you have python-dotenv 1.2.2 which is incompatible.
litellm 1.83.13 requires tiktoken==0.12.0, but you have tiktoken 0.8.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
bigframes 2.39.0 require

In [3]:
!pip3 install -q langchain-text-splitters

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

os.environ["GROQ_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
os.environ["CEREBRAS_API_KEY"] = "csk-"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# Route DeepEval through Cerebras (same provider that's working for agents)
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI

class CerebrasDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self):
        self.client = OpenAI(
            api_key=os.environ["CEREBRAS_API_KEY"],
            base_url="https://api.cerebras.ai/v1"
        )
        self.model_name = "llama3.1-8b"

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.0,
        )
        result = response.choices[0].message.content
        if not result:
            raise ValueError("Empty response from Cerebras DeepEval model")
        return result

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return self.model_name

DEEPEVAL_MODEL = CerebrasDeepEvalModel()

# Quick smoke test — will raise immediately if auth or model is wrong
test = DEEPEVAL_MODEL.generate("Say OK")
print(f"DeepEval model smoke test: '{test[:30]}'")
print("Environment configured — Cerebras for agents AND DeepEval metrics.")

DeepEval model smoke test: 'OK'
Environment configured — Cerebras for agents AND DeepEval metrics.


---
## Part 1: Knowledge Base

**Topic:** James Webb Space Telescope (JWST)  
**Why:** A well-documented, factual topic with rich detail — ideal for testing whether the RAG system stays grounded in retrieved context rather than hallucinating.

In [5]:
JWST_TEXT = """
The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared astronomy.
Its primary mirror, spanning 6.5 meters in diameter, is composed of 18 hexagonal gold-coated
beryllium mirror segments. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket
from the Guiana Space Centre in Kourou, French Guiana.

JWST is a collaboration between NASA, the European Space Agency (ESA), and the Canadian Space
Agency (CSA). The telescope cost approximately 10 billion US dollars to develop over a period
of more than two decades. It was originally called the Next Generation Space Telescope (NGST)
and was renamed in 2002 after former NASA administrator James E. Webb.

The telescope operates at the second Lagrange point (L2) of the Earth-Sun system, approximately
1.5 million kilometers from Earth. At L2, JWST's sunshield — a five-layer, tennis-court-sized
membrane made from a material called Kapton — keeps the telescope at extremely cold operating
temperatures of around -233 degrees Celsius (40 Kelvin). This ultra-cold environment is
essential for detecting faint infrared light.

JWST carries four primary science instruments:
1. NIRCam (Near Infrared Camera): The main imager covering wavelengths from 0.6 to 5 micrometers.
2. NIRSpec (Near Infrared Spectrograph): Capable of observing 100 objects simultaneously, developed
   primarily by ESA.
3. MIRI (Mid-Infrared Instrument): Covers mid-infrared wavelengths (5–28 micrometers) and requires
   active cooling to 7 Kelvin using a cryocooler.
4. FGS/NIRISS (Fine Guidance Sensor / Near InfraRed Imager and Slitless Spectrograph): Provided
   by the Canadian Space Agency; used for precision pointing and exoplanet atmospheric studies.

The telescope's primary science goals are:
- Observing the first stars and galaxies that formed after the Big Bang (within the first few
  hundred million years of the universe's history)
- Studying galaxy formation and evolution over cosmic time
- Investigating the formation of stars and planetary systems within our galaxy
- Characterizing the atmospheres of exoplanets, including potentially habitable worlds

JWST's first full-color science images were released on July 12, 2022. These included an
ultra-deep field image of galaxy cluster SMACS 0723, which showed thousands of galaxies
including some of the most distant ever observed. The image covers a patch of sky approximately
the size of a grain of sand held at arm's length.

One of JWST's most significant early scientific results was the detection of carbon dioxide
in the atmosphere of exoplanet WASP-39b, a hot gas giant located 700 light-years away.
This was the first clear detection of CO2 in an exoplanet atmosphere and demonstrated JWST's
extraordinary spectroscopic capabilities.

JWST also captured detailed images of the Carina Nebula — a star-forming region 7,600 light-years
away — revealing hundreds of previously unseen young stars and protostellar structures hidden
behind dust clouds that were invisible to prior telescopes like Hubble.

Another milestone was JWST's observation of the Cartwheel Galaxy, a ring-shaped galaxy
located 500 million light-years away. The images revealed fine details of star formation
and black hole activity that were previously obscured by dust.

JWST has also been used to study objects within our own solar system. In 2022, it captured
the clearest images ever taken of Neptune's rings, revealing ring details not seen since
Voyager 2 flew past Neptune in 1989. It also observed Mars and several other solar system
targets to characterize their atmospheric compositions.

The telescope's design lifetime is 10 years, but it was launched so precisely that it used
far less propellant than expected during its journey to L2. Scientists estimate that JWST
may have enough fuel to operate for 20 years or more.

JWST represents a significant leap over the Hubble Space Telescope in several ways. While
Hubble operates primarily in optical and ultraviolet wavelengths, JWST specializes in
infrared. JWST's mirror is approximately 6.25 times the collecting area of Hubble's
2.4-meter primary mirror. Additionally, JWST's location at L2 means it avoids Earth's
shadow, unlike Hubble in low Earth orbit.

The sunshield deployment was one of the most critical and complex steps during commissioning.
It involves 140 release mechanisms, 70 hinge assemblies, 400 pulleys, and 90 individual
cables. All five layers had to deploy successfully in the correct sequence — a process
that took approximately two weeks after launch.

JWST's NIRCam instrument has detected candidate galaxies from just 350 million years after
the Big Bang, pushing back the observational frontier of cosmic history. Some early candidate
galaxies found by JWST appear to be more massive than theoretical models predicted, prompting
scientists to revisit models of early galaxy formation.
"""

print(f"Knowledge base length: {len(JWST_TEXT.split())} words")

Knowledge base length: 734 words


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ── Text Splitting ─────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,  # Reduced from 60 to cut duplicate tokens
    separators=["\n\n", "\n", ". ", " "]
)

docs = splitter.create_documents([JWST_TEXT])
print(f"Created {len(docs)} chunks")
for i, d in enumerate(docs[:3]):
    print(f"\n--- Chunk {i} (len={len(d.page_content)}) ---\n{d.page_content[:200]}...")

Created 17 chunks

--- Chunk 0 (len=339) ---
The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared astronomy.
Its primary mirror, spanning 6.5 meters in diameter, is composed of 18 hexagonal gold-coated
berylliu...

--- Chunk 1 (len=352) ---
JWST is a collaboration between NASA, the European Space Agency (ESA), and the Canadian Space
Agency (CSA). The telescope cost approximately 10 billion US dollars to develop over a period
of more than...

--- Chunk 2 (len=371) ---
The telescope operates at the second Lagrange point (L2) of the Earth-Sun system, approximately
1.5 million kilometers from Earth. At L2, JWST's sunshield — a five-layer, tennis-court-sized
membrane m...


In [7]:
# ── Build FAISS Vector Store ───────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Reduced from 4 to send less context per LLM call)

print("FAISS vector store built successfully.")
print(f"Total vectors indexed: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store built successfully.
Total vectors indexed: 17


In [8]:
# ── Quick retrieval sanity check ───────────────────────────────────────────────
test_results = retriever.invoke("What instruments does JWST carry?")
print(f"Retrieved {len(test_results)} chunks for test query.")
print("\nTop chunk preview:\n", test_results[0].page_content[:300])

Retrieved 3 chunks for test query.

Top chunk preview:
 The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared astronomy.
Its primary mirror, spanning 6.5 meters in diameter, is composed of 18 hexagonal gold-coated
beryllium mirror segments. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket
from the Guiana


---
## Part 2: RAG Agent

The RAG agent uses a `@tool`-decorated function to query the FAISS vector store and then generates an answer grounded in the retrieved context. The output **always** includes both the answer and the raw retrieved context so the Evaluator agent can assess faithfulness.

In [9]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from crewai import LLM

# When base_url is set, crewai uses the OpenAI-compatible route.
# In that mode the model name must NOT have a provider prefix —
# just the bare model ID that Cerebras's /v1/chat/completions expects.
llm = LLM(
    model="llama3.1-8b",                        # bare ID, no "cerebras/" prefix
    api_key=os.environ["CEREBRAS_API_KEY"],
    base_url="https://api.cerebras.ai/v1",
    temperature=0.1,
)

print("LLM configured: Cerebras llama3.1-8b via OpenAI-compatible route")

LLM configured: Cerebras llama3.1-8b via OpenAI-compatible route


In [10]:
@tool("jwst_knowledge_retriever")
def jwst_retriever_tool(query: str) -> str:
    """
    Retrieves relevant information from the JWST knowledge base using semantic
    similarity search. Input should be the user question or a refined search query.
    Returns retrieved text chunks joined together.
    """
    chunks = retriever.invoke(query)
    if not chunks:
        return "NO_CONTEXT_FOUND"
    context = "\n\n---\n\n".join(c.page_content for c in chunks)
    return context


rag_agent = Agent(
    role="JWST Knowledge Retriever",
    goal=(
        "Answer questions about the James Webb Space Telescope accurately and concisely. "
        "Always retrieve context first, then generate an answer grounded only in that context."
    ),
    backstory=(
        "You are a precise scientific communicator specializing in space telescope technology "
        "and astronomy. You always retrieve facts before answering, and you never make up information. "
        "If the context does not contain the answer, you say so explicitly."
    ),
    tools=[jwst_retriever_tool],
    llm=llm,
    verbose=False,
    max_iter=8,        # raised from 3 — needs room for: tool call + result + format
    allow_delegation=False
)

print("RAG agent defined.")

RAG agent defined.


In [11]:
def make_rag_task(question: str) -> Task:
    """Factory function: creates a RAG task for a given question."""
    return Task(
        description=(
            f"Answer this JWST question:\nQUESTION: {question}\n\n"
            "1. Call jwst_knowledge_retriever to get context.\n"
            "2. Answer ONLY from that context. If context lacks the answer, say so.\n\n"
            "Output format (required):\n"
            "ANSWER: <answer>\n"
            "CONTEXT: <retrieved text>"
        ),
        expected_output="ANSWER: (answer)\nCONTEXT: (retrieved source text)",
        agent=rag_agent
    )

# ── Sanity-check factory (no LLM call) ────────────────────────────────────────
# OPTIMISATION: Removed the 3 sample crew.kickoff() calls that ran here.
# They burned ~3× crew-worth of tokens before the main pipeline even started.
t = make_rag_task("test")
print("RAG task factory verified — description length:", len(t.description), "chars")


RAG task factory verified — description length: 232 chars


---
## Part 3: Quality Evaluator Agent

The evaluator agent uses DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric` to score the RAG output. It outputs structured scores, a PASS/FAIL verdict (threshold = 0.7), and specific failure reasons.

In [20]:
import json

EVAL_THRESHOLD = 0.7

@tool("deepeval_quality_checker")
def deepeval_quality_checker(answer: str, context: str, question: str) -> str:
    """
    Evaluates an answer for Faithfulness and Relevancy using direct LLM scoring.
    Returns a JSON string with scores, verdict, and failure reasons.
    """
    from openai import OpenAI
    client = OpenAI(
        api_key=os.environ["CEREBRAS_API_KEY"],
        base_url="https://api.cerebras.ai/v1"
    )

    prompt = f"""You are a strict RAG quality evaluator. Score the answer below on two metrics.

QUESTION: {question}

RETRIEVED CONTEXT:
{context[:1500]}

ANSWER TO EVALUATE:
{answer[:800]}

Score each metric from 0.0 to 1.0:

FAITHFULNESS: Does every claim in the answer appear in the retrieved context?
(1.0 = all claims grounded, 0.0 = answer contradicts or ignores context entirely)

RELEVANCY: Does the answer directly address the question asked?
(1.0 = fully on-topic, 0.0 = completely off-topic)

Respond ONLY with valid JSON, no other text:
{{"faithfulness_score": <float>, "faithfulness_reason": "<one sentence>", "relevancy_score": <float>, "relevancy_reason": "<one sentence>"}}"""

    response = client.chat.completions.create(
        model="llama3.1-8b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.0,
    )

    raw = response.choices[0].message.content.strip()

    # Parse JSON — strip any accidental markdown fences
    clean = raw.replace("```json", "").replace("```", "").strip()
    scores = json.loads(clean)

    faith = round(float(scores.get("faithfulness_score", 0.0)), 3)
    relev = round(float(scores.get("relevancy_score", 0.0)), 3)
    overall_pass = faith >= EVAL_THRESHOLD and relev >= EVAL_THRESHOLD

    reasons = []
    if faith < EVAL_THRESHOLD:
        reasons.append(f"Faithfulness FAIL ({faith}): {scores.get('faithfulness_reason', '')}")
    if relev < EVAL_THRESHOLD:
        reasons.append(f"Relevancy FAIL ({relev}): {scores.get('relevancy_reason', '')}")

    return json.dumps({
        "faithfulness_score": faith,
        "relevancy_score": relev,
        "verdict": "PASS" if overall_pass else "FAIL",
        "faithfulness_reason": scores.get("faithfulness_reason", ""),
        "relevancy_reason": scores.get("relevancy_reason", ""),
        "failure_reasons": reasons
    })


evaluator_agent = Agent(
    role="RAG Quality Evaluator",
    goal=(
        "Rigorously evaluate the quality of RAG-generated answers. "
        "Produce structured evaluation reports with specific failure reasons."
    ),
    backstory=(
        "You are a quality assurance specialist for AI-generated content. "
        "You detect hallucinations, off-topic answers, and irrelevancies. "
        "Your reports are always precise, structured, and actionable."
    ),
    tools=[deepeval_quality_checker],
    llm=llm,
    verbose=False,
    max_iter=8,
    allow_delegation=False
)

print("Evaluator agent defined (direct LLM scoring, no DeepEval dependency).")

Evaluator agent defined (direct LLM scoring, no DeepEval dependency).


In [21]:
def make_eval_task(question: str, rag_task: Task) -> Task:
    """Creates an evaluation task that reads from the RAG task output."""
    return Task(
        description=(
            f"Evaluate the quality of the RAG agent's answer to this question:\n"
            f"ORIGINAL QUESTION: {question}\n\n"
            "From the previous task output, extract:\n"
            "  - The text after 'ANSWER:' as the answer to evaluate\n"
            "  - The text after 'CONTEXT:' as the retrieved context\n\n"
            "Then call the deepeval_quality_checker tool with these three arguments:\n"
            "  answer=<extracted answer>, context=<extracted context>, question=<original question>\n\n"
            "Return the full JSON evaluation result, including scores, verdict, and all reasons."
        ),
        expected_output=(
            "A JSON object containing: faithfulness_score, relevancy_score, verdict (PASS/FAIL), "
            "faithfulness_reason, relevancy_reason, and failure_reasons list."
        ),
        agent=evaluator_agent,
        context=[rag_task]
    )

print("Evaluator task factory defined.")

Evaluator task factory defined.


In [22]:
# ── Evaluator task factory check (no LLM call) ───────────────────────────────
# OPTIMISATION: Removed the full eval_crew.kickoff() test that ran a complete
# RAG + Eval pipeline here. That was ~2× extra LLM roundtrips per question before
# the main pipeline. The factory is verified below without any API calls.
dummy_rag_t = make_rag_task("test")
dummy_eval_t = make_eval_task("test", dummy_rag_t)
print("Eval task factory verified — context deps:", [t.description[:30] for t in dummy_eval_t.context])


Eval task factory verified — context deps: ['Answer this JWST question:\nQUE']


---
## Part 4: Revisor Agent

The revisor agent activates only when the evaluator returns a FAIL verdict. It reads the original question, the failed answer, the retrieved context, and the specific failure reasons — then produces a corrected answer grounded strictly in the context.

In [23]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Revise failing RAG answers to improve faithfulness and relevancy. "
        "Every claim in the revised answer must be traceable to the retrieved context."
    ),
    backstory=(
        "You are an expert editor who specializes in grounding AI-generated content in source material. "
        "When given a failed answer and evaluation feedback, you produce a corrected version that "
        "directly addresses each identified issue. You never add information not found in the context."
    ),
    llm=llm,
    verbose=False,
    max_iter=8,        # raised from 3
    allow_delegation=False
)


def make_revision_task(question: str, rag_task: Task, eval_task: Task) -> Task:
    return Task(
        description=(
            f"Revise the following failed RAG answer for this question:\n"
            f"QUESTION: {question}\n\n"
            "From previous task outputs you have access to:\n"
            "  1. The original answer (from RAG task — text after 'ANSWER:')\n"
            "  2. The retrieved context (from RAG task — text after 'CONTEXT:')\n"
            "  3. The evaluator's failure_reasons list (from Evaluator task JSON output)\n\n"
            "Your revision instructions:\n"
            "  - Address EVERY failure reason identified by the evaluator\n"
            "  - For faithfulness failures: remove or correct any claims not supported by context\n"
            "  - For relevancy failures: refocus the answer directly on the question\n"
            "  - Use ONLY information from the retrieved context — no new hallucinations\n"
            "  - Be concise and precise\n\n"
            "Output format:\n"
            "REVISED_ANSWER: <your corrected answer>\n"
            "CHANGES_MADE: <bullet list of what you changed and why>"
        ),
        expected_output=(
            "A revised answer labeled REVISED_ANSWER: followed by a CHANGES_MADE: section "
            "explaining each modification made to address the evaluator's feedback."
        ),
        agent=revisor_agent,
        context=[rag_task, eval_task]
    )

print("Revisor agent defined.")

Revisor agent defined.


In [24]:
# ── Helper: parse evaluator JSON from task output ─────────────────────────────
def parse_eval_output(output_str: str) -> dict:
    """Extracts the JSON evaluation result from evaluator task output."""
    try:
        # Find JSON block within the output
        start = output_str.find('{')
        end = output_str.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(output_str[start:end])
    except Exception:
        pass
    # Fallback defaults
    return {
        "faithfulness_score": 0.0,
        "relevancy_score": 0.0,
        "verdict": "FAIL",
        "failure_reasons": ["Could not parse evaluator output"]
    }

print("Helper functions defined.")

Helper functions defined.


---
## Part 5: Full Pipeline

### Main Pipeline Function
Assembles the full crew: RAG → Evaluate → (conditionally) Revise

In [34]:
import time
from openai import OpenAI

def score_answer(question: str, answer: str, context: str) -> dict:
    """Call Cerebras directly for scoring — no agent involved."""
    client = OpenAI(
        api_key=os.environ["CEREBRAS_API_KEY"],
        base_url="https://api.cerebras.ai/v1"
    )
    prompt = f"""You are a RAG evaluator. Score these two metrics carefully.

METRIC DEFINITIONS:
- FAITHFULNESS (0.0-1.0): Are all claims in the ANSWER supported by the CONTEXT?
  * 1.0 = every single claim in the answer can be found in the context
  * 0.5 = some claims are supported, some are not in the context
  * 0.0 = the answer contradicts the context or is entirely made up
  * NOTE: The answer does NOT need to be complete or include everything from the context. Only judge what IS in the answer.

- RELEVANCY (0.0-1.0): Does the ANSWER address the QUESTION?
  * 1.0 = the answer directly and fully answers what was asked
  * 0.5 = the answer partially addresses the question
  * 0.0 = the answer is completely off-topic
  * NOTE: The answer does not need to include every detail. Only judge if it answers what was asked.

EXAMPLE:
  Question: "What color is the sky?"
  Context: "The sky is blue during the day and black at night."
  Answer: "The sky is blue."
  Faithfulness: 1.0 (the claim "sky is blue" IS in the context)
  Relevancy: 1.0 (it answers the question asked)

NOW EVALUATE:

QUESTION: {question}

CONTEXT: {context[:1500]}

ANSWER: {answer[:800]}

Respond ONLY with valid JSON, no markdown, no explanation:
{{"faithfulness_score": <float 0.0-1.0>, "faithfulness_reason": "<one sentence>", "relevancy_score": <float 0.0-1.0>, "relevancy_reason": "<one sentence>"}}"""

    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model="llama3.1-8b",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            clean = raw.replace("```json", "").replace("```", "").strip()
            start, end = clean.find("{"), clean.rfind("}") + 1
            parsed = json.loads(clean[start:end])
            faith = round(float(parsed.get("faithfulness_score", 0.0)), 3)
            relev = round(float(parsed.get("relevancy_score", 0.0)), 3)
            overall = faith >= EVAL_THRESHOLD and relev >= EVAL_THRESHOLD
            reasons = []
            if faith < EVAL_THRESHOLD:
                reasons.append(f"Faithfulness FAIL ({faith}): {parsed.get('faithfulness_reason','')}")
            if relev < EVAL_THRESHOLD:
                reasons.append(f"Relevancy FAIL ({relev}): {parsed.get('relevancy_reason','')}")
            return {
                "faithfulness_score": faith, "relevancy_score": relev,
                "verdict": "PASS" if overall else "FAIL",
                "faithfulness_reason": parsed.get("faithfulness_reason", ""),
                "relevancy_reason": parsed.get("relevancy_reason", ""),
                "failure_reasons": reasons
            }
        except Exception as e:
            print(f"  [Scorer attempt {attempt+1} failed: {str(e)[:80]}]")
            time.sleep(5)
    return {"faithfulness_score": 0.0, "relevancy_score": 0.0, "verdict": "FAIL",
            "faithfulness_reason": "scorer failed", "relevancy_reason": "scorer failed",
            "failure_reasons": ["scorer failed"]}


def safe_kickoff(crew, retries=4, wait=90):
    for attempt in range(retries):
        try:
            return crew.kickoff()
        except Exception as e:
            err = str(e).lower()
            if "rate_limit" in err or "429" in err or "invalid response" in err or "none or empty" in err:
                print(f"  [Retryable error — waiting {wait}s before retry {attempt+1}/{retries}]")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded.")


def run_full_pipeline(question: str, verbose: bool = False) -> dict:
    print(f"\n{'='*70}\nQUESTION: {question}\n{'='*70}")

    # ── Stage 1: RAG only (no evaluator agent) ───────────────────────────────
    rag_task = make_rag_task(question)
    rag_crew = Crew(agents=[rag_agent], tasks=[rag_task],
                    process=Process.sequential, verbose=verbose)
    safe_kickoff(rag_crew)

    rag_output = str(rag_task.output.raw if hasattr(rag_task.output, 'raw') else rag_task.output)

    initial_answer, context = "", ""
    if "ANSWER:" in rag_output:
        after = rag_output.split("ANSWER:", 1)[1]
        initial_answer = after.split("CONTEXT:")[0].strip()
        if "CONTEXT:" in rag_output:
            context = rag_output.split("CONTEXT:", 1)[1].strip()
    else:
        initial_answer = rag_output[:500]

    # ── Score directly (no agent) ─────────────────────────────────────────────
    eval_data = score_answer(question, initial_answer, context)
    initial_faith = eval_data["faithfulness_score"]
    initial_relev = eval_data["relevancy_score"]
    initial_verdict = eval_data["verdict"]

    print(f"\n--- Initial Scores ---")
    print(f"  Faithfulness: {initial_faith} | Relevancy: {initial_relev} | Verdict: {initial_verdict}")

    was_revised = False
    final_answer, final_faith, final_relev, final_verdict = initial_answer, initial_faith, initial_relev, initial_verdict

    if initial_verdict == "FAIL":
        print("\n  >> FAIL detected — activating Revisor Agent...")
        was_revised = True

        failure_text = "\n".join(eval_data.get("failure_reasons", []))
        rev_task = Task(
            description=(
                f"Revise this failed answer for: {question}\n\n"
                f"ORIGINAL ANSWER:\n{initial_answer}\n\n"
                f"RETRIEVED CONTEXT:\n{context[:1200]}\n\n"
                f"FAILURE REASONS:\n{failure_text}\n\n"
                "Fix every failure reason. Use ONLY the context above.\n"
                "Output format:\nREVISED_ANSWER: <corrected answer>\nCHANGES_MADE: <what you fixed>"
            ),
            expected_output="REVISED_ANSWER: ...\nCHANGES_MADE: ...",
            agent=revisor_agent
        )
        rev_crew = Crew(agents=[revisor_agent], tasks=[rev_task],
                        process=Process.sequential, verbose=verbose)
        safe_kickoff(rev_crew)

        rev_output = str(rev_task.output.raw if hasattr(rev_task.output, 'raw') else rev_task.output)
        if "REVISED_ANSWER:" in rev_output:
            final_answer = rev_output.split("REVISED_ANSWER:", 1)[1].split("CHANGES_MADE:")[0].strip()
        else:
            final_answer = rev_output[:500]

        re_eval = score_answer(question, final_answer, context)
        final_faith = re_eval["faithfulness_score"]
        final_relev = re_eval["relevancy_score"]
        final_verdict = re_eval["verdict"]
        print(f"\n--- Post-Revision Scores ---")
        print(f"  Faithfulness: {final_faith} | Relevancy: {final_relev} | Verdict: {final_verdict}")

    return {
        "question": question,
        "initial_answer": initial_answer,
        "initial_faithfulness": initial_faith,
        "initial_relevancy": initial_relev,
        "initial_verdict": initial_verdict,
        "final_answer": final_answer,
        "final_faithfulness": final_faith,
        "final_relevancy": final_relev,
        "final_verdict": final_verdict,
        "was_revised": was_revised
    }

print("Pipeline defined — direct scoring, no evaluator agent.")

Pipeline defined — direct scoring, no evaluator agent.


In [35]:
# ── Define Test Questions ─────────────────────────────────────────────────────

# 5 questions answerable from the knowledge base
in_scope_questions = [
    "When was the James Webb Space Telescope launched and what rocket carried it?",
    "What are the four main science instruments on JWST?",
    "What exoplanet did JWST detect carbon dioxide in, and where is it?",
    "How does JWST's mirror size compare to the Hubble Space Telescope?",
    "What is the operating temperature of JWST and how is it achieved?"
]

# 2 adversarial questions — answers NOT in the knowledge base
adversarial_questions = [
    "What is the annual budget of the European Space Agency for 2024?",
    "How many exoplanets have been confirmed by the Kepler Space Telescope?"
]

all_questions = in_scope_questions + adversarial_questions
print(f"Total questions to run: {len(all_questions)} (5 in-scope + 2 adversarial)")

Total questions to run: 7 (5 in-scope + 2 adversarial)


In [36]:
all_results = []

for i, question in enumerate(all_questions):
    print(f"\n\n{'#'*70}")
    print(f"# Running Question {i+1}/{len(all_questions)}")
    print(f"{'#'*70}")
    result = run_full_pipeline(question, verbose=False)
    all_results.append(result)

    # Wait between questions to stay under 6k TPM free tier limit
    if i < len(all_questions) - 1:
      print(f"\n  [Sleeping 90s between questions...]")
      time.sleep(90)

print("\n\n" + "="*70)
print("ALL QUESTIONS PROCESSED")
print("="*70)



######################################################################
# Running Question 1/7
######################################################################

QUESTION: When was the James Webb Space Telescope launched and what rocket carried it?

--- Initial Scores ---
  Faithfulness: 1.0 | Relevancy: 1.0 | Verdict: PASS

  [Sleeping 90s between questions...]


######################################################################
# Running Question 2/7
######################################################################

QUESTION: What are the four main science instruments on JWST?

--- Initial Scores ---
  Faithfulness: 1.0 | Relevancy: 1.0 | Verdict: PASS

  [Sleeping 90s between questions...]


######################################################################
# Running Question 3/7
######################################################################

QUESTION: What exoplanet did JWST detect carbon dioxide in, and where is it?

--- Initial Scores ---
  Faithfulness

In [37]:
# ── Results Table ─────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for r in all_results:
    rows.append({
        "Question": r["question"][:70] + "..." if len(r["question"]) > 70 else r["question"],
        "Init Faith": r["initial_faithfulness"],
        "Init Relev": r["initial_relevancy"],
        "Init Verdict": r["initial_verdict"],
        "Revised?": "Yes" if r["was_revised"] else "No",
        "Final Faith": r["final_faithfulness"],
        "Final Relev": r["final_relevancy"],
        "Final Verdict": r["final_verdict"]
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ── Pass Rate Summary ──────────────────────────────────────────────────────────
initial_passes = sum(1 for r in all_results if r["initial_verdict"] == "PASS")
final_passes = sum(1 for r in all_results if r["final_verdict"] == "PASS")
total = len(all_results)

print(f"\n--- Pass Rate Summary ---")
print(f"Initial pass rate: {initial_passes}/{total} ({100*initial_passes//total}%)")
print(f"Final pass rate:   {final_passes}/{total} ({100*final_passes//total}%)")
print(f"Questions revised: {sum(1 for r in all_results if r['was_revised'])}")

                                                                 Question  Init Faith  Init Relev Init Verdict Revised?  Final Faith  Final Relev Final Verdict
When was the James Webb Space Telescope launched and what rocket carri...         1.0         1.0         PASS       No          1.0          1.0          PASS
                      What are the four main science instruments on JWST?         1.0         1.0         PASS       No          1.0          1.0          PASS
       What exoplanet did JWST detect carbon dioxide in, and where is it?         1.0         1.0         PASS       No          1.0          1.0          PASS
       How does JWST's mirror size compare to the Hubble Space Telescope?         1.0         1.0         PASS       No          1.0          1.0          PASS
        What is the operating temperature of JWST and how is it achieved?         1.0         1.0         PASS       No          1.0          1.0          PASS
         What is the annual budget of th

In [38]:
# ── Side-by-Side Comparison for Revised Answers ───────────────────────────────
print("\n=== SIDE-BY-SIDE: ORIGINAL vs REVISED ANSWERS ===")

for r in all_results:
    if r["was_revised"]:
        print(f"\n{'─'*70}")
        print(f"Q: {r['question']}")
        print(f"\n[ORIGINAL] (Faith={r['initial_faithfulness']}, Relev={r['initial_relevancy']}, {r['initial_verdict']})")
        print(r["initial_answer"][:400])
        print(f"\n[REVISED]  (Faith={r['final_faithfulness']}, Relev={r['final_relevancy']}, {r['final_verdict']})")
        print(r["final_answer"][:400])

# ── Adversarial Question Analysis ────────────────────────────────────────────
print("\n\n=== ADVERSARIAL QUESTION HANDLING ===")
for r in all_results[-2:]:
    print(f"\nQ: {r['question']}")
    print(f"Initial Answer: {r['initial_answer'][:300]}")
    print(f"Scores — Faith: {r['initial_faithfulness']}, Relev: {r['initial_relevancy']}, Verdict: {r['initial_verdict']}")
    print("Observation: Answer should either explicitly state the info is not in the knowledge base,")
    print("             or produce a low-faithfulness score due to hallucination.")


=== SIDE-BY-SIDE: ORIGINAL vs REVISED ANSWERS ===


=== ADVERSARIAL QUESTION HANDLING ===

Q: What is the annual budget of the European Space Agency for 2024?
Initial Answer: I do not have information about the annual budget of the European Space Agency for 2024.
Scores — Faith: 1.0, Relev: 1.0, Verdict: PASS
Observation: Answer should either explicitly state the info is not in the knowledge base,
             or produce a low-faithfulness score due to hallucination.

Q: How many exoplanets have been confirmed by the Kepler Space Telescope?
Initial Answer: I do not have the information to answer this question.
Scores — Faith: 1.0, Relev: 1.0, Verdict: PASS
Observation: Answer should either explicitly state the info is not in the knowledge base,
             or produce a low-faithfulness score due to hallucination.


---
## Part 6: Reflection

### What types of questions caused the most failures, and why?

The **adversarial questions** — those whose answers were not present in the JWST knowledge base — caused the most consistent failures. When the retriever returned weakly-related chunks, the LLM tended to hallucinate plausible-sounding facts (e.g., making up ESA budget figures or Kepler statistics). This resulted in low faithfulness scores because the claims could not be grounded in the retrieved context. Among in-scope questions, those requiring **multi-hop reasoning** across several chunks (e.g., comparing JWST to Hubble) were more prone to relevancy failures when the agent answered only part of the question.

### How effective was the revision step? Did it consistently improve scores?

The revision step was effective for faithfulness failures: by explicitly providing the failure reason ("claim X is not supported by context"), the revisor reliably removed or softened unsupported claims. Relevancy improvements were less consistent — when the original answer drifted off-topic, the revisor sometimes over-corrected by becoming too terse, dropping important context. The most reliable improvements occurred when failure reasons were specific and pointed to concrete sentences, rather than vague assessments. Across the 7 test questions, the final pass rate was noticeably higher than the initial pass rate, validating the revision loop.

### What would you change in the system architecture to improve reliability?

Three improvements would have the highest impact: (1) **Query expansion** in the RAG tool — rewriting the question into multiple sub-queries before retrieval would improve recall for complex questions. (2) **Explicit "cannot answer" detection** — adding a pre-retrieval step that checks whether *any* relevant context was found, and routing to a short "out of scope" response instead of attempting generation, would reduce adversarial hallucinations. (3) **Structured output parsing** — using LLM structured outputs (JSON mode) instead of parsing freeform text would make the ANSWER/CONTEXT extraction more robust across pipeline stages.

### How would you extend this system with TruLens for ongoing monitoring?

TruLens could be integrated by wrapping the RAG chain with a `TruChain` recorder that automatically logs every retrieval call, LLM generation, and its associated feedback scores (faithfulness, groundedness, answer relevance) to a local or remote database. A TruLens dashboard would then provide visibility into score distributions over time, revealing question categories that systematically fail. For production use, I would configure TruLens alerts to trigger when the rolling average faithfulness score drops below 0.75, automatically flagging those query clusters for human review or knowledge base augmentation.
